In [ ]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2


In [ ]:
from langchain_openai import AzureChatOpenAI

from databricks_asset_bundle.large_language_model.lc_model import SummarizationModel

llm = AzureChatOpenAI(model="gpt-4.1")

trainer = SummarizationModel(llm)
model = trainer.train()


In [ ]:
input_examples = [
    {
        "text": """
        **The Heart of the Jungle**

The sun hung low in the sky, casting long shadows over the dense foliage as Captain Elara Hayes stepped into the unknown. Her heart raced with a mix of trepidation and exhilaration; she was about to embark on a journey that few had dared to attempt. The tales of the uncharted jungle had lured her in like a siren’s call, whispering secrets of lost civilizations and treasures untold. But it was not gold or jewels that fueled her ambition; it was the thrill of discovery and the chance to write her own story in the annals of exploration.

Elara had spent months preparing for this expedition. Armed with a compass, a weathered journal, and a sturdy machete, she felt ready to face whatever the jungle might throw at her. Her guides, a group of local indigenous people familiar with the jungle’s whispers and dangers, stood behind her, their expressions unreadable. They had warned her about the spirits that haunted the trees and the creatures that roamed the underbrush, but Elara was undeterred. She had come too far to turn back now.

As the group plunged deeper into the greenery, the air thickened with humidity and the scent of damp earth. Towering trees loomed overhead, their branches entwined like ancient fingers reaching for the sky. Sunlight filtered through the leaves, creating a mosaic of light and shadow on the forest floor. The sounds of the jungle enveloped her: the distant call of a howler monkey, the rustle of unseen creatures, and the hum of insects that buzzed like a living orchestra.

Hours turned into days as they navigated the labyrinthine paths of the jungle. Elara kept meticulous notes, sketching maps and documenting every detail of the flora and fauna they encountered. She was captivated by vibrant orchids that bloomed like jewels and towering ferns that seemed to whisper secrets of a bygone era. But as enchanting as the jungle was, it was also filled with dangers. The group faced torrential rains that turned paths into rivers, insects that left painful welts on their skin, and the ever-present threat of venomous snakes.

One evening, as they set up camp near a bubbling stream, Elara felt a chill in the air. The locals grew silent, their eyes darting toward the shadows. It was then that she heard it—a faint melody drifting through the trees. It was a haunting tune, unlike anything she had ever heard. Intrigued, Elara followed the sound, her feet moving almost of their own accord.

“Captain, wait!” one of her guides called, but she was already lost in the rhythm of the music. The notes seemed to beckon her deeper into the jungle, and despite the warnings echoing in her mind, she felt an irresistible pull.

As she ventured further, the jungle transformed. The trees grew thicker, their trunks twisted and gnarled, and the underbrush became a dense tangle of roots and vines. The melody intensified, wrapping around her like a warm embrace. Suddenly, she emerged into a clearing, and her breath caught in her throat.

Before her stood a colossal stone structure, half-swallowed by the jungle. Vines draped over its weathered stones, and moss clung to its surface like a green blanket. It was a temple, ancient and majestic, a remnant of a civilization long forgotten. Elara’s heart raced with excitement; this was a discovery that would change everything.

As she approached the temple, she noticed intricate carvings adorning its walls. They depicted scenes of rituals, celestial bodies, and creatures that seemed to come alive under the dappled sunlight. The melody grew louder, echoing off the stone and filling the air with an otherworldly energy. Elara reached out to touch the carvings, her fingers tracing the lines that had been etched by hands long gone.

Suddenly, a rustle in the bushes made her spin around. From the shadows emerged a figure, cloaked in the colors of the jungle. The figure stepped forward, revealing a woman with a crown of leaves and flowers, her skin adorned with intricate tattoos that spoke of her lineage.

“You should not be here,” the woman said, her voice a mix of authority and sadness. “The jungle protects its secrets.”

Elara stood tall, her determination unwavering. “I mean no harm. I am an explorer, seeking to understand and learn. I wish to share your story with the world.”

The woman studied her for a moment, and Elara held her breath, wondering if she would be cast out or worse. Finally, the woman sighed, and the tension in her shoulders eased. “If you seek knowledge, then follow me, but know this: some stories are not meant to be told.”

With a nod, Elara followed the woman into the temple. Inside, the air was cool and fragrant with the scent of earth and decay. They walked through narrow hallways adorned with more carvings, each one revealing the rich history of the people who had once thrived here. The woman spoke of their ancestors, their rituals, and the bond they shared with the jungle.

As Elara listened, she realized the importance of preserving these stories—not just for her own glory, but for the sake of the people who had come before. The jungle was alive, and its heart beat in rhythm with the memories it held. When they emerged from the temple, dusk had begun to settle, and the jungle transformed into a symphony of twilight sounds.

“Thank you for sharing this with me,” Elara said, her voice filled with reverence.

The woman smiled, a glimmer of hope in her eyes. “Take this knowledge back with you, but remember: the jungle is not just a place to explore. It is a living entity, deserving of respect.”

With her guides waiting anxiously at the edge of the clearing, Elara returned, her heart full of purpose. She had ventured into the uncharted jungle seeking treasure, but she had discovered something far more valuable—the wisdom of a culture intertwined with nature. As she made her way back to civilization, she knew that her journey was just beginning. She would tell their story, not as an outsider, but as a bridge between worlds, honoring the heart of the jungle and the people who called it home.
        """,
    }
]


In [ ]:
summaries = model.batch(input_examples)


In [ ]:
model_name = dbutils.widgets.get("summarization_model_name")  # noqa: F821


In [ ]:
from logging import getLogger

import mlflow
import mlflow.langchain
from mlflow.models import infer_signature

log = getLogger(__name__)

with mlflow.start_run() as run:
    log.info("Active run ID: %s", run.info.run_id)
    model_info = mlflow.langchain.log_model(
        lc_model=model,
        artifact_path="model",
        signature=infer_signature(input_examples, summaries),
    )
    log.info("Model URI: %s", model_info.model_uri)
    mlflow.register_model(
        model_uri=model_info.model_uri,
        name=model_name,
        tags={"model_type": "langchain"},
    )
